# Handwritten Digit Classification using Deep Learning

## Problem Description
The MNIST dataset contains 70,000 grayscale images of handwritten digits (0-9). Each image is 28x28 pixels. The goal is to build a deep learning model that can accurately classify these handwritten digits.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
axes = axes.ravel()
for i in range(10):
    idx = np.where(y_train == i)[0][0]
    axes[i].imshow(X_train[idx], cmap='gray')
    axes[i].set_title(f'Digit: {i}')
    axes[i].axis('off')
plt.suptitle('Sample Images from Each Class')
plt.tight_layout()
plt.show()

In [ ]:
digit_counts = pd.Series(y_train).value_counts().sort_index()
plt.figure(figsize=(10, 6))
plt.bar(digit_counts.index, digit_counts.values)
plt.xlabel('Digit')
plt.ylabel('Count')
plt.title('Distribution of Digits in Training Set')
plt.xticks(range(10))
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"\nClass distribution in training set:")
for digit, count in digit_counts.items():
    print(f"Digit {digit}: {count} samples ({count/len(y_train)*100:.1f}%)")

In [ ]:
pixel_means = X_train.mean(axis=0)
plt.figure(figsize=(8, 8))
plt.imshow(pixel_means, cmap='hot')
plt.colorbar()
plt.title('Average Pixel Intensities Across All Training Images')
plt.show()

In [ ]:
sample_images = X_train[:5]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, (img, label) in enumerate(zip(sample_images, y_train[:5])):
    axes[i].hist(img.ravel(), bins=50, color='blue', alpha=0.7)
    axes[i].set_title(f'Pixel Distribution - Digit {label}')
    axes[i].set_xlabel('Pixel Value')
    axes[i].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train_cat, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Training set shape: {X_train_split.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

## Model Architecture

In [ ]:
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Model Training

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001)

history = model.fit(
    X_train_split, y_train_split,
    batch_size=128,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Set Performance:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Detailed Performance Analysis

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred_classes))

In [ ]:
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
misclassified_idx = np.where(y_pred_classes != y_test)[0]
print(f"Number of misclassified samples: {len(misclassified_idx)} out of {len(y_test)}")

fig, axes = plt.subplots(2, 5, figsize=(12, 6))
axes = axes.ravel()
for i in range(min(10, len(misclassified_idx))):
    idx = misclassified_idx[i]
    axes[i].imshow(X_test[idx].reshape(28, 28), cmap='gray')
    axes[i].set_title(f'True: {y_test[idx]}, Pred: {y_pred_classes[idx]}')
    axes[i].axis('off')
plt.suptitle('Examples of Misclassified Images')
plt.tight_layout()
plt.show()

In [ ]:
confidence_correct = []
confidence_incorrect = []
for i in range(len(y_test)):
    confidence = np.max(y_pred[i])
    if y_pred_classes[i] == y_test[i]:
        confidence_correct.append(confidence)
    else:
        confidence_incorrect.append(confidence)

plt.figure(figsize=(10, 6))
plt.hist(confidence_correct, bins=50, alpha=0.7, label=f'Correct Predictions (n={len(confidence_correct)})', density=True)
plt.hist(confidence_incorrect, bins=50, alpha=0.7, label=f'Incorrect Predictions (n={len(confidence_incorrect)})', density=True)
plt.xlabel('Prediction Confidence')
plt.ylabel('Density')
plt.title('Prediction Confidence Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Comparison with Simple Neural Network

In [ ]:
model_simple = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model_simple.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_simple = model_simple.fit(
    X_train_split, y_train_split,
    batch_size=128,
    epochs=20,
    validation_data=(X_val, y_val),
    verbose=0
)

simple_loss, simple_accuracy = model_simple.evaluate(X_test, y_test_cat, verbose=0)

print(f"Simple NN Test Accuracy: {simple_accuracy:.4f}")
print(f"CNN Test Accuracy: {test_accuracy:.4f}")
print(f"Improvement: {(test_accuracy - simple_accuracy)*100:.2f}%")
print(f"CNN parameters: {model.count_params():,}")
print(f"Simple NN parameters: {model_simple.count_params():,}")

## Discussion and Conclusions

### 1. Model Performance
- The CNN achieved excellent accuracy on the test set (typically ~99%)
- Training converged smoothly with minimal overfitting
- Early stopping prevented overtraining

### 2. Error Analysis
- Most misclassifications occur between visually similar digits (e.g., 4 and 9, 3 and 8)
- Incorrect predictions generally have lower confidence scores
- The confusion matrix shows that the model performs well across all digit classes

### 3. Model Architecture
- 3 convolutional layers with max pooling effectively extracted hierarchical features
- Dropout layer (0.5) helped prevent overfitting
- The model is relatively small and efficient

### 4. Comparison with Baseline
- CNN significantly outperforms a simple fully connected neural network
- The improvement demonstrates the effectiveness of convolutional layers for image data

### 5. Potential Improvements
- Data augmentation (rotation, scaling, shifting) could improve generalization
- Ensemble methods might reduce the error rate further
- More advanced architectures (ResNet blocks, attention mechanisms) could capture more complex patterns
- Hyperparameter tuning could optimize performance